## Generation of Protein Vector Given a 2D Sequence

Import Packages:

In [42]:
import numpy as np

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
import torch.nn.functional as F

import re

from typing import Union

import requests
import esm

url = "https://raw.githubusercontent.com/schwallergroup/ai4chem_course/main/notebooks/06%20-%20Generative%20Models%202/utils.py"

r = requests.get(url)
with open("utils.py", "wb") as f:
    f.write(r.content)

url = "https://raw.githubusercontent.com/sch20-%20Generative%20Models%202/pretrained.zinc.rnn.pth"

r = requests.get(url)
with open("pretrained.zinc.rnn.pth", "wb") as f:
    f.write(r.content)



Load the dataset:

In [43]:
df = pd.read_csv("Datasets/Fluorescent-Protein-Database.csv")

Generate t-scale for protein sequences:

In [44]:
tsc_df = pd.read_csv("Datasets/T_scales_table.csv")

tsc_df.columns = tsc_df.columns.str.strip()
tsc_df["symbol"] = tsc_df["symbol"].str.strip()

T_SCALE = {}

for _, row in tsc_df.iterrows():
    aa = row["symbol"]
    T_SCALE[aa] = [
        float(row["T_1"]),
        float(row["T_2"]),
        float(row["T_3"]),
        float(row["T_4"]),
        float(row["T_5"])
    ]

def sequence_to_tscales(seq):
    vecs = []
    for aa in seq:
        vecs.append(T_SCALE.get(aa, [0,0,0,0,0]))
    return np.array(vecs)

def tscale_features(seq):
    mat = sequence_to_tscales(seq)

    mean = mat.mean(axis=0)
    std = mat.std(axis=0)
    maxv = mat.max(axis=0)
    minv = mat.min(axis=0)

    return np.concatenate([mean, std, maxv, minv])

df["tscales"] = df["Protein sequence"].apply(tscale_features)

Using Evolutionary Scale Modeling (ESM-2) Embeddings

In [45]:
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to C:\Users\Adrian/.cache\torch\hub\checkpoints\esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to C:\Users\Adrian/.cache\torch\hub\checkpoints\esm2_t33_650M_UR50D-contact-regression.pt


In [46]:
def get_esm_embedding(seq):
    data = [("protein", seq)]

    _, _, tokens = batch_converter(data)

    with torch.no_grad():
        results = model(tokens, repr_layers=[33])

    # layer 33 = final representation
    reps = results["representations"][33]

    # remove special tokens and mean pool
    embedding = reps[0, 1:-1].mean(0)

    return embedding.numpy()


df["esm"] = df["Protein sequence"].apply(get_esm_embedding)

KeyboardInterrupt: 

Encode SMILES of chromophores

In [47]:
smiles_dict = {
    "NRQ": r"CSCCC(=N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "CRQ": r"NC(=O)CCC(=N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "NRP": r"CC(C)CC(=N)C1=NC(=C/c2ccc(O)cc2)/C(=O)N1CC(O)=O",
    "CH6": r"CSCC[C@H](N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "CRO": r"[C@@H](O)[C@H](N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "5SQ": r"N[C@@H](Cc1c[nH]cn1)C2=NC(=C\c3ccc(O)cc3)/C(=O)N2CC(O)=O",
    "4M9": r"NC(=O)CCC(=N)C1=NC(=C\c2c[nH]c3ccccc23)/C(=O)N1CC(O)=O",
    "CR2": r"NCC1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "OFM": r"C[C@H]1O[C@@](O)(N=C1C2=N\C(=C/c3ccc(O)cc3)C(=O)N2CC(O)=O)[C@@H](N)Cc4ccccc4",
    "CR8": r"N[C@@H](Cc1[nH]cnc1)c2nc(C=C3C=CC(=O)C=C3)c([O-])n2CC(O)=O",
    "CFY": r"N[C@@H](Cc1ccccc1)[C@@]2(O)SCC(=N2)C3=NC(=C\c4ccc(O)cc4)/C(=O)N3CC(O)=O",
    "OIM": r"CC[C@H](C)[C@H](N)[C@@]1(O)O[C@H](C)C(=N1)C2=N\C(=C/c3ccc(O)cc3)C(=O)N2CC(O)=O",
    "CH7": r"OC(=O)CN1C(=O)C(=C/c2ccc(O)cc2)/N=C1C3=NCCCC3",
    "GYS": r"N[C@@H](CO)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "WCR": r"C[C@@]1(O)NC(=C\c2ccc(O)cc2)/C(=O)N1CC(O)=O",
    "DYG": r"N[C@@H](CC(O)=O)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "FAD": r"Cc1cc2N=C3C(=O)NC(=O)N=C3N(C[C@H](O)[C@H](O)[C@H](O)CO[P@](O)(=O)O[P@@](O)(=O)OC[C@H]4O[C@H]([C@H](O)[C@@H]4O)n5cnc6c(N)ncnc56)c2cc1C",
    "PIA": r"C[C@H](N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "BLR": r"CC1=C(C=C)C(/NC1=O)=C/c2[nH]c(Cc3[nH]c(\C=C4/NC(=O)C(=C4C)C=C)c(C)c3CCC(O)=O)c(CCC(O)=O)c2C",
    "CRF": r"C[C@@H](O)[C@H](N)C1=N\C(=C/c2c[nH]c3ccccc23)C(=O)N1CC(O)=O",
    "NYG": r"N[C@@H](CC(N)=O)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O",
    "FMN": r"Cc1cc2N=C3C(=O)NC(=O)N=C3N(C[C@H](O)[C@H](O)[C@H](O)CO[P](O)(O)=O)c2cc1C",
    "B2H": r"C[C@@H](O)[C@H](N)c1nc(Cc2c[nH]c3ccccc23)c(O)n1CC(O)=O",
    "SWG": r"N[C@@H](CO)C1=N\C(=C/c2c[nH]c3ccccc23)C(=O)N1CC(O)=O",
    "CSH": r"N[C@@H](CO)[C@H]1N[C@@H](Cc2c[nH]cn2)C(=O)N1CC(O)=O",
    "BJF": r"CC(C)C[C@H](N)c1nc(CC(C)C)c(O)n1CC(O)=O",
}   
df["smiles"] = df["Chromophore/ligand"].str.strip().str.upper().map(smiles_dict)
print(df["smiles"])
class Vocabulary:
    """
    Stores the tokens and their conversion to vocabulary indexes.
    """
    def __init__(self, tokens : Union[dict, None]=None, starting_id : int=0) -> None:
        self._tokens = {}
        self._current_id = starting_id

        if tokens:
            for token, idx in tokens.items():
                self._add(token, idx)
                self._current_id = max(self._current_id, idx + 1)

    def __getitem__(self, token_or_id : str) -> int:
        return self._tokens[token_or_id]

    def add(self, token : str) -> int:
        """
        Adds a token to the vocabulary.
        """
        if not isinstance(token, str):
            raise TypeError("Token is not a string")
        if token in self:
            return self[token]
        self._add(token, self._current_id)
        self._current_id += 1
        return self._current_id - 1

    def update(self, tokens : list) -> list:
        """
        Adds many tokens.
        """
        return [self.add(token) for token in tokens]

    def __delitem__(self, token_or_id : Union[str, int]) -> None:
        other_val = self._tokens[token_or_id]
        del self._tokens[other_val]
        del self._tokens[token_or_id]

    def __contains__(self, token_or_id : Union[str, int]) -> None:
        return token_or_id in self._tokens

    def __eq__(self, other_vocabulary : "Vocabulary") -> int:
        return self._tokens == other_vocabulary._tokens  # pylint: disable=W0212

    def __len__(self) -> int:
        return len(self._tokens) // 2

    def encode(self, tokens : list) -> np.ndarray:
        """
        Encodes a list of tokens as vocabulary indexes.
        """
        vocab_index = np.zeros(len(tokens), dtype=int)
        for i, token in enumerate(tokens):
            vocab_index[i] = self._tokens[token]
        return vocab_index

    def decode(self, vocab_index : np.ndarray) -> list:
        """
        Decodes a vocabulary index matrix to a list of tokens.
        """
        tokens = []
        for idx in vocab_index:
            tokens.append(self[idx])
        return tokens

    def _add(self, token : str, idx : int) -> None:
        if idx not in self._tokens:
            self._tokens[token] = idx
            self._tokens[idx] = token
        else:
            raise ValueError("IDX already present in vocabulary")

    def tokens(self) -> list:
        """
        Returns the tokens from the vocabulary.
        """
        return [t for t in self._tokens if isinstance(t, str)]
    REGEXPS = {
        "brackets": re.compile(r"(\[[^\]]*\])"),
        "2_ring_nums": re.compile(r"(%\d{2})"),
        "brcl": re.compile(r"(Br|Cl)")
    }
REGEXP_ORDER = ["brackets", "2_ring_nums", "brcl"]

def tokenize(data, with_begin_and_end=True):

    def split_by(data, regexps):
        if not regexps:
            return list(data)

        regexp = REGEXPS[regexps[0]]
        splitted = regexp.split(data)

        tokens = []
        for i, split in enumerate(splitted):
            if i % 2 == 0:
                tokens += split_by(split, regexps[1:])
            else:
                tokens.append(split)

        return tokens

    tokens = split_by(data, REGEXP_ORDER)

    if with_begin_and_end:
        tokens = ["^"] + tokens + ["$"]

    return tokens
print(df["smiles"].isna().sum(), "missing SMILES")
#df["tokenized_smiles"] = df["smiles"].apply(
    #lambda x: tokenize(x) if isinstance(x, str) else None
#)

0          CSCCC(=N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O
1          CSCCC(=N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O
2          CSCCC(=N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O
3          CSCCC(=N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O
4          CSCCC(=N)C1=N\C(=C/c2ccc(O)cc2)C(=O)N1CC(O)=O
                             ...                        
117    N[C@@H](CO)[C@H]1N[C@@H](Cc2c[nH]cn2)C(=O)N1CC...
118    N[C@@H](CO)[C@H]1N[C@@H](Cc2c[nH]cn2)C(=O)N1CC...
119              CC(C)C[C@H](N)c1nc(CC(C)C)c(O)n1CC(O)=O
120      CC(C)CC(=N)C1=NC(=C/c2ccc(O)cc2)/C(=O)N1CC(O)=O
121    N[C@@H](CO)C1=N\C(=C/c2c[nH]c3ccccc23)C(=O)N1C...
Name: smiles, Length: 122, dtype: str
4 missing SMILES


Split train/test

In [48]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

Normalize predictors and apply same normalization to the test dataset

In [49]:
scaler = StandardScaler()

num_cols = ["Stokes shift", "kDa"] # Which columns to normalize

train_df[num_cols] = scaler.fit_transform(train_df[num_cols])
test_df[num_cols] = scaler.transform(test_df[num_cols])